# Initializing kcat values for PAMparametrizer

In this notebook, we give an overview of how an initial set of turnover number can be obtained. To enable this, we use multiple databases to map kcat values obtained from machine learning (from the [GotEnzymes](https://metabolicatlas.org/gotenzymes) database) to the proteins and reactions in the model using the gene-protein-reaction associations.

Before we start, we have to set up all the package and files required. Adjust the filenames and paths as desired.

In [1]:
import pandas as pd
import os
from cobra.io import read_sbml_model
import matplotlib.pyplot as plt
import re
import numpy as np
import datetime

from Bio.KEGG import REST
import requests

if os.path.split(os.getcwd())[1]=='Scripts':
    os.chdir('..')

DEFAULT_PROT_MW = 39959.4825 #g/mol
DEFAULT_KCAT =  11.0 #/s median value from BRENDA
DEFAULT_KCAT_TRANSPORT = 100000000 #/s

PAM_DATA_FILE_PATH = os.path.join('Data', 'proteinAllocationModel_iJN1463_EnzymaticData.xlsx')
METANETX_FILE_PATH = os.path.join('Data', 'Databases', 'reac_xref.tsv') #mapping rxn to EC number
mnx_chem_xref= pd.read_csv(os.path.join('Data', 'Databases',"chem_xref.tsv"), sep="\t",comment="#") #mapping chemicals
UNIPROT_FILE_PATH = os.path.join('Data', 'Databases','uniprotkb_pputidakt2240_240807.xlsx')
RHEA2KEGG_FILE_PATH = os.path.join('Data', 'Databases','rhea2kegg_reaction.tsv')

GOTENZYMES_FILE_PATH = os.path.join('Data','Databases', 'GotEnzymes_ppu_240807.json')

In [2]:
#output file path
# Create a datetime object for the current date
current_date = datetime.datetime.now()
# Format the date as yymmdd
formatted_date = current_date.strftime('%y%m%d')

OUTPUT_FILE_PATH = os.path.join('Data', f'proteinAllocationModel_iJN1463_EnzymaticData_{formatted_date}.xlsx')
AE_OUTPUT_FILE_PATH = os.path.join('Data', f'{formatted_date}_ppu_ActiveEnzymeInformation_GotEnzymes.xlsx')

## 0. Download information from different databases
1. **Metabolic model stoichiometries and reactions: [BiGG database](http://bigg.ucsd.edu/)**
- Get your model! Or use your own reaction identifiers
- In this example we will use the iJN1463 model for [*Pseudomonas putida* KT2240](http://bigg.ucsd.edu/models/iJN1463)
- Important: the identifiers must me mappable to uniprot (or another enzyme identifier database) and KEGG (for mapping to GotEnzymes)

2. **Turnover numbers: [GotEnzymes](https://metabolicatlas.org/gotenzymes)**
- Go to the [API of the Metabolic Atlas](https://metabolicatlas.org/api/v2/#/GotEnzymes/gotEnzymes)
- Use the `GET\gotenzymes\enzymes?` functionality, click `Try it out`
- Here we filled in `ppu` (*P. putida* strain KT2240) for `organism` and `10000` for `pagination[pageSize]`. The rest was left empty
- Click execute and a json file with all the information will be generated
- Once the webpage is finished processing your query, scroll down and copy/download the resulting json file
- Note: GotEnzymes gives the locus tag, ec number, reaction id and compound for each protein. The latter 2 are given as KEGG identifiers, which we subsequently have to map to BiGG identifiers in order to match with the model. Therefore, we need to use other databases

3. **Gene-Protein-Reaction relations: [UniprotKB](https://www.uniprot.org/)**
- Got to the [Uniprot knowledge base](https://www.uniprot.org/) and used advanced search to query for your organism. For *P. putida* we used [this](https://www.uniprot.org/uniprotkb?query=%22pseudomonas+putida%22+AND+%28organism_id%3A160488%29) (strain: *Pseudomonas putida (strain ATCC 47054 / DSM 6125 / CFBP 8728 / NCIMB 11950 / KT2440)*) webpage
- Click download, select `format`:`Excel` and select the following: 
    - `Uniprot Data -> Names & Taxonomy`: `Gene Names`
    - `Uniprot Data -> Sequences`: `length`, `mass`
    - `Uniprot Data -> Function`: `RheaID`
- Mark `Compressed` as `No`
- Download the resulting Excel file
    
4. **Match KEGG reaction ids to Uniprot protein ids: [Rhea Database](https://www.rhea-db.org/)**
- Download the `rhea2kegg_reaction.tsv` file from the Cross-Reference section of the [download page of the rhea website](https://www.rhea-db.org/help/download)

5. **Match KEGG reaction ids to BIGG reaction ids: [MetaNetX](https://www.metanetx.org/)**
- Download the `chem_xref.tsv` and `reac_xref.tsv` flatfiles from the [MetaNetX namespace](https://www.metanetx.org/mnxdoc/mnxref.html)

## 1. Get information from the BiGG model

In the metabolic model itself is already quite some information stored. In order to obtain up-to-date information on the enzymes related to the reaction, we will need to obtain the kegg reaction ID, the reactants, the product and the gene-protein relation.

In [3]:
#load the model
model = read_sbml_model(os.path.join('Models', 'iJN1463.xml'))

expand=False

#make a id mapping and create a mapping dataframe
id_mapper_df = pd.DataFrame(columns = ['rxn_id', 'kegg_id', 'rhea_id', 'mnx_id','Reactants','Products','EC', 'GPR'])
for rxn in model.reactions:
    #skip transport reactions and ATP maintenance requirement
    if 'ex' in rxn.id or 'EX' in rxn.id or rxn.id == 'ATPM':
        continue
    to_append= []   
    to_append.append(rxn.annotation['bigg.reaction'])
    #not all reactions are annotated with a KEGG ID in the model
    try: 
        to_append.append(rxn.annotation['kegg.reaction'])
    except:
        to_append.append(np.nan)
    try: 
        to_append.append(rxn.annotation['rhea'])
    except:
        to_append.append(np.nan)
    try: 
        to_append.append(rxn.annotation['metanetx.reaction'])
    except:
        to_append.append(np.nan)
    try:
        to_append.append([reac.annotation['kegg.compound'] for reac in rxn.reactants])
    except:
        to_append.append([])
    try: 
        to_append.append([prod.annotation['kegg.compound'] for prod in rxn.products])
    except:
        to_append.append([])
    if 'ec-code' in rxn.annotation.keys():
        #if there are multiple enzymes make seperate lists
        if isinstance(rxn.annotation['ec-code'], list) and expand:
            info = to_append.copy()
            for i in range(len(rxn.annotation['ec-code'])):
                #make a new row for each enzyme by copying the previous information
                info = to_append.copy()
                ec= rxn.annotation['ec-code'][i]
                info.append(ec)
                #add gpr
                info.append(rxn.gene_reaction_rule)
                #add to df
                id_mapper_df.loc[len(id_mapper_df)] = info
        else:
            to_append.append(rxn.annotation['ec-code'])
            to_append.append(rxn.gene_reaction_rule)
            id_mapper_df.loc[len(id_mapper_df)] = to_append
    else:
        to_append.append(np.nan)
        #ignore entries without enzyme and gpr relation
        if rxn.gene_reaction_rule =='':
            continue
        else:
            to_append.append(rxn.gene_reaction_rule)
        id_mapper_df.loc[len(id_mapper_df)] = to_append
        
#remove entries without GRP relation or Ec number


print('Number of reactions in the mapping dataframe: ',len(id_mapper_df))
print('Number of reactions mapped to KEGG identifier: ', len(id_mapper_df.kegg_id.dropna()))
print('Number of reactions mapped to rhea identifier: ', len(id_mapper_df.rhea_id.dropna()))
print('Number of reactions mapped to MetaNetX identifier: ', len(id_mapper_df.mnx_id.dropna()))
id_mapper_df.head()

Set parameter Username
Academic license - for non-commercial use only - expires 2025-03-06
Number of reactions in the mapping dataframe:  2007
Number of reactions mapped to KEGG identifier:  698
Number of reactions mapped to rhea identifier:  777
Number of reactions mapped to MetaNetX identifier:  1375


,rxn_id,kegg_id,rhea_id,mnx_id,Reactants,Products,EC,GPR
0,3HAD160,R04544,NaN,MNXR94881,[C04633],"[[C00001, C01328], C05763]","[2.3.1.85, 2.3.1.86, 4.2.1.61, 4.2.1.59, 2.3.1.-]",PP_1602 or PP_4174
1,1P2CBXLCYCL,NaN,NaN,NaN,[C01110],[],NaN,PP_s0001
2,1P2CBXLR,NaN,NaN,NaN,[],"[C00006, [C00148, C16435]]",NaN,PP_3591
3,1PPDCRc,R02203,"[12525, 12526, 12527, 12524]",MNXR94711,"[C04092, C00080, C00005]","[C00408, C00006]","[1.5.1.1, 1.5.1.25, 1.5.1.21]",PP_3591
4,1PY4h3cAH,NaN,NaN,NaN,[],[],NaN,PP_1257


## 2. Get missing kegg ids from MetaNetX

Normally, there are quite some reaction identifiers without KEGG annotation in the model. Therefore, we use the static information from MetaNetX to fill in some of the gaps

In [4]:
#load the mnx dataframe
mnx_reac_xref= pd.read_csv(METANETX_FILE_PATH, sep="\t",comment="#")
mnx_reac_xref.columns = ['ex_id', 'mnx_id', 'equation']
mnx_reac_xref

,ex_id,mnx_id,equation
0,bigg.reaction:CRBNTD,EMPTY,H2CO3 dissociation||1 biggM:h2co3@biggC:x = 1 ...
1,bigg.reaction:DNADRAIN,EMPTY,Dna sink transport reaction|| =
2,bigg.reaction:H2CO3D2,EMPTY,Carboxylic acid dissociation||1 biggM:h2co3@bi...
3,bigg.reaction:H2CO3D2m,EMPTY,"Carboxylic acid dissociation, mitochondrial||1..."
4,bigg.reaction:HMR_5409,EMPTY,HMR 5409||1 biggM:h@biggC:e + 1 biggM:hco3@big...
...,...,...,...
384796,biggR:R_GALNACT3g,MNXR99998,secondary/obsolete/fantasy identifier
384797,bigg.reaction:GALNACT4g,MNXR99999,Uridine diphosphoacetylgalactosamine-chondroit...
384798,bigg.reaction:R_GALNACT4g,MNXR99999,secondary/obsolete/fantasy identifier
384799,biggR:GALNACT4g,MNXR99999,Uridine diphosphoacetylgalactosamine-chondroit...


In [5]:
#parse the bigg and kegg ids to separate dataframes
mnx_reac_bigg = pd.DataFrame(columns=['mnx_id', 'bigg_id', 'bigg_substrates', 'bigg_products'])
mnx_reac_kegg = pd.DataFrame(columns=['mnx_id', 'kegg_id', 'substrates', 'products'])


for index, row in mnx_reac_xref.iterrows():
    #find bigg id using regex search
    bigg_pattern = re.search(r"bigg.reaction:(\w+)", row['ex_id'])
    if bigg_pattern and row['mnx_id']!= 'EMPTY': #check if there is a BiGG id and a mnx id
        index = len(mnx_reac_bigg)
        mnx_reac_bigg.loc[index, 'bigg_id'] = bigg_pattern.group(1)
        mnx_reac_bigg.loc[index, 'mnx_id'] = row['mnx_id']
        #if there is a equation available, extract the substrates and products into separate columns
        if 'biggM' in row['equation']:
            bigg_equation = row['equation'].split("||") #extract the equation part (last entry)
            equation_splitted = bigg_equation[-1].split('=') #split substrates and products
            
            #make a list of the substrates and products using a regex search
            substrates = [re.findall(" biggM:(.*)@biggC", str(compound)) for compound in equation_splitted[0].split('+')]
            products = [re.findall(" biggM:(.*)@biggC", str(compound)) for compound in equation_splitted[1].split('+')]
            
            #unlisting the individual compounds and skip empty entries
            mnx_reac_bigg.loc[index,'bigg_substrates'] = [comp[0] for comp in substrates if comp!= []]
            mnx_reac_bigg.loc[index,'bigg_products'] = [comp[0] for comp in products if comp!= []]
            
    #find kegg id using regex search
    kegg_pattern = re.search(r"kegg.reaction:(\w+)", row['ex_id'])
    if kegg_pattern:
        index = len(mnx_reac_kegg)
        mnx_reac_kegg.loc[index, 'kegg_id'] = kegg_pattern.group(1)
        mnx_reac_kegg.loc[index, 'mnx_id'] = row['mnx_id']
        #if there is a equation available, extract the substrates and products into separate columns
        if isinstance(row['equation'], str):
            kegg_equation = row['equation'].split("||") #extract the equation part (last entry)
            equation_splitted = kegg_equation[-1].split('=') #split substrates and products
            
            #make a list of the substrates and products using a regex search
            substrates = [re.findall(" keggC:(.*)@IN", str(compound)) for compound in equation_splitted[0].split('+')]
            products = [re.findall(" keggC:(.*)@IN", str(compound)) for compound in equation_splitted[1].split('+')]
            
            #unlisting the individual compounds and skip empty entries
            mnx_reac_kegg.loc[index,'substrates'] = [comp[0] for comp in substrates if comp!= []]
            mnx_reac_kegg.loc[index,'products'] = [comp[0] for comp in products if comp!= []]
            
print(mnx_reac_bigg)
print(mnx_reac_kegg)

          mnx_id      bigg_id      bigg_substrates        bigg_products
0         MNXR02       EX_h_e                   []                  [h]
1         MNXR02     R_EX_h_e                  NaN                  NaN
2         MNXR03     HMR_1095                  [h]                  [h]
3         MNXR03           Ht                  [h]                  [h]
4         MNXR03         Htcx                  [h]                  [h]
...          ...          ...                  ...                  ...
56321  MNXR99997  R_GALNACT2g                  NaN                  NaN
56322  MNXR99998    GALNACT3g  [h, cs_c_pre3, udp]  [cs_c_pre2, uacgam]
56323  MNXR99998  R_GALNACT3g                  NaN                  NaN
56324  MNXR99999    GALNACT4g  [h, cs_d_pre4, udp]  [cs_d_pre3, uacgam]
56325  MNXR99999  R_GALNACT4g                  NaN                  NaN

[56326 rows x 4 columns]
           mnx_id kegg_id                substrates                  products
0      MNXR100024  R00253  [C000

In [6]:
#there are roughly 5x more entries from BiGG than from KEGG
print('Number of entries from BiGG ', len(mnx_reac_bigg))
print('Number of entries from KEGG ', len(mnx_reac_kegg))

Number of entries from BiGG  56326
Number of entries from KEGG  11333


In [7]:
#match the bigg and kegg ids based on mnx id
mnx_reac_bigg_kegg = mnx_reac_bigg.merge(mnx_reac_kegg, on = 'mnx_id', how = 'inner')

print('Number of reaction bigg-mnx-kegg pairs ', len(mnx_reac_bigg_kegg))

Number of reaction bigg-mnx-kegg pairs  6712


In [8]:
kegg_matches_before = len(id_mapper_df.kegg_id.dropna())

#add the information to the id_mapper_df
for index, row in id_mapper_df.iterrows():
    #check if kegg dataframe is not known
    if row['kegg_id'] == '':
        #is there a match available from the metanetx data?
        bigg_match_kegg = mnx_reac_bigg_kegg['bigg_id'] == row['rxn_id']
        if sum(bigg_match_kegg)>0:
            #fill in the rows with the kegg info
            matches=mnx_reac_bigg_kegg[bigg_match_kegg]
            id_mapper_df.loc[index, 'kegg_id'] = matches['kegg_id'].iloc[0]
            id_mapper_df.loc[index, 'Reactants'] = matches['substrates'].iloc[0]
            id_mapper_df.loc[index, 'Products'] = matches['products'].iloc[0]
            
kegg_matches_after = len(id_mapper_df.kegg_id.dropna())
            
print('Number of reactions mapped to KEGG rxnid went from ', kegg_matches_before, ' to ', kegg_matches_after)

Number of reactions mapped to KEGG rxnid went from  698  to  698


### 2.1. Try to improve matching by matching bigg metabolites ids to kegg compound ids

In [9]:
#1. load chemical compound xref data
mnx_chem_xref.columns = ['ex_id', 'mnx_id', 'info']
mnx_chem_xref

,ex_id,mnx_id,info
0,mnx:BIOMASS,BIOMASS,BIOMASS
1,seed.compound:cpd11416,BIOMASS,Biomass
2,seedM:M_cpd11416,BIOMASS,secondary/obsolete/fantasy identifier
3,seedM:cpd11416,BIOMASS,Biomass
4,MNXM01,MNXM01,PMF||Translocated proton that acccounts for th...
...,...,...,...
2996504,sabiork.compound:40,WATER,H2O||Water
2996505,sabiorkM:40,WATER,H2O||Water
2996506,seed.compound:cpd00001,WATER,H2O||H20||H3O+||HO-||Hydroxide ion||OH||OH-||W...
2996507,seedM:M_cpd00001,WATER,secondary/obsolete/fantasy identifier


In [10]:
# parse the bigg and kegg ids to separate dataframes
mnx_chem_bigg = pd.DataFrame(columns=['mnx_id', 'bigg_id'])
mnx_chem_kegg = pd.DataFrame(columns=['mnx_id', 'kegg_id'])


for index, row in mnx_chem_xref.iterrows():
    #find bigg id using regex search
    bigg_pattern = re.search(r"bigg.metabolite:(\w+)", row['ex_id'])
    if bigg_pattern and row['mnx_id']!= 'EMPTY': #check if there is a BiGG id and a mnx id
        index = len(mnx_chem_bigg)
        mnx_chem_bigg.loc[index, 'bigg_id'] = bigg_pattern.group(1)
        mnx_chem_bigg.loc[index, 'mnx_id'] = row['mnx_id']
        
    #find kegg id using regex search
    kegg_pattern = re.search(r"kegg.compound:(\w+)", row['ex_id'])
    if kegg_pattern:
        index = len(mnx_chem_kegg)
        mnx_chem_kegg.loc[index, 'kegg_id'] = kegg_pattern.group(1)
        mnx_chem_kegg.loc[index, 'mnx_id'] = row['mnx_id']
            
print(mnx_chem_bigg)
print(mnx_chem_kegg)

         mnx_id      bigg_id
0        MNXM02          oh1
1         MNXM1            h
2        MNXM10         nadh
3       MNXM100         grdp
4     MNXM10053  mercplaccys
...         ...          ...
9082    MNXM988        34hpl
9083    MNXM990     3htmelys
9084    MNXM992       4mptnl
9085   MNXM9931      35cdamp
9086      WATER          h2o

[9087 rows x 2 columns]
          mnx_id kegg_id
0         MNXM02  C01328
1          MNXM1  C00080
2         MNXM10  C00004
3        MNXM100  C00341
4      MNXM10006  C16496
...          ...     ...
18806   MNXM9986  C06057
18807   MNXM9987  C05145
18808   MNXM9989  C16720
18809   MNXM9990  C16266
18810      WATER  C00001

[18811 rows x 2 columns]


In [11]:
# mnx_reac_bigg.to_csv(os.join(BASEDIR, 'Results', 'mnx_reac_bigg.csv'))
kegg_matches = id_mapper_df[id_mapper_df['Products'].str.len()>0]
kegg_matches_before = len(kegg_matches[kegg_matches['Reactants'].str.len()>0])

In [12]:
# # match compound information to the id_mapper_df
# kegg_matches = id_mapper_df[id_mapper_df['Products'].str.len()>0]
# kegg_matches_before = len(kegg_matches[kegg_matches['Reactants'].str.len()>0])

# #add the information to the id_mapper_df
# for index, row in id_mapper_df.iterrows():
#     #check if kegg dataframe is not known
#     if row['Products'] == [] and row['Reactants'] == []:
#         #is there a match available from the metanetx data?
#         bigg_match = mnx_reac_bigg['bigg_id'] == row['rxn_id']
#         if sum(bigg_match)>0:
#             bigg_row = mnx_reac_bigg[bigg_match]
#             #fill in the rows with the kegg info
#             kegg_products = []
#             for product in bigg_row['bigg_products'].iloc[0]:
#                 mnx_chem_id= mnx_chem_bigg[mnx_chem_bigg['bigg_id'] == product]
#                 kegg_chem_id =mnx_chem_kegg[mnx_chem_kegg['mnx_id'] == mnx_chem_id]['kegg_id']
#                 kegg_products += [kegg_chem_id.iloc[0]]
#             kegg_substrates = []
                
#             for substrate in bigg_row['bigg_substrates'].iloc[0]:
#                 mnx_chem_id= mnx_chem_bigg[mnx_chem_bigg['bigg_id'] == substrate]
#                 kegg_chem_id =mnx_chem_kegg[mnx_chem_kegg['mnx_id'] == mnx_chem_id]['kegg_id']
#                 kegg_substrates += [kegg_chem_id.iloc[0]]
                
            
#             id_mapper_df.loc[index, 'Reactants'] = kegg_substrates
#             id_mapper_df.loc[index, 'Products'] = kegg_products
            
# kegg_matches = id_mapper_df[id_mapper_df['Products'].str.len()>0]
# kegg_matches_after = len(kegg_matches[kegg_matches['Reactants'].str.len()>0])
            
# print('Number of reactions mapped to KEGG compound ids went from ', kegg_matches_before, ' to ', kegg_matches_after)

## 4. Get all the protein information

Besides information about the reactions, we need the relation to the proteins in order to establish a protein allocation model. We do this using the gene-protein-reaction association obtained from the BIGG model. The genes is the model can be mapped to the Uniprot accessions, which can be mapped to the KEGG reactions using the Rhea database.

### 4.1 Parse the BIGG gene-protein-reaction associations

In [13]:
#You need to adjust this to find the geneid/locustag for your microbe
locustag_regex =r'\b(PP_\d{4})\b'
def extract_locustag_numbers(text):
    return re.findall(locustag_regex, text)

id_mapper_df['b_number'] = id_mapper_df['GPR'].apply(extract_locustag_numbers)
id_mapper_df = id_mapper_df.explode('b_number', ignore_index=True)
id_mapper_df

,rxn_id,kegg_id,rhea_id,mnx_id,Reactants,Products,EC,GPR,b_number
0,3HAD160,R04544,NaN,MNXR94881,[C04633],"[[C00001, C01328], C05763]","[2.3.1.85, 2.3.1.86, 4.2.1.61, 4.2.1.59, 2.3.1.-]",PP_1602 or PP_4174,PP_1602
1,3HAD160,R04544,NaN,MNXR94881,[C04633],"[[C00001, C01328], C05763]","[2.3.1.85, 2.3.1.86, 4.2.1.61, 4.2.1.59, 2.3.1.-]",PP_1602 or PP_4174,PP_4174
2,1P2CBXLCYCL,NaN,NaN,NaN,[C01110],[],NaN,PP_s0001,NaN
3,1P2CBXLR,NaN,NaN,NaN,[],"[C00006, [C00148, C16435]]",NaN,PP_3591,PP_3591
4,1PPDCRc,R02203,"[12525, 12526, 12527, 12524]",MNXR94711,"[C04092, C00080, C00005]","[C00408, C00006]","[1.5.1.1, 1.5.1.25, 1.5.1.21]",PP_3591,PP_3591
...,...,...,...,...,...,...,...,...,...
3760,4HTHRS,R05086,"[30659, 30662, 30660, 30661]",MNXR95024,"[[C00001, C01328], C06055]","[C06056, C00009]","[4.2.3.1, 4.2.3]",PP_1471 or PP_0662,PP_0662
3761,4MBZALDH,R05282,NaN,MNXR95030,"[C06757, C00003]","[C06758, C00080, C00004]","[1.1.1.90, 1.1.1.21]",pWW0_128,NaN
3762,4MBZDH,NaN,NaN,MNXR95031,"[C06758, [C00001, C01328], C00003]","[C00080, C00004, C01454]","[1.2.1.3, 1.2.1.28, 1.2.1.7]",pWW0_131,NaN
3763,4MCAT23DOX,R05295,NaN,MNXR95032,"[C06730, C00007]","[C00080, C06760]",1.13.11.2,pWW0_097,NaN


### 4.2 Parse the Uniprot data for merging

In [14]:
# load the uniprot information
uniprot_df = pd.read_excel(UNIPROT_FILE_PATH)
#get the gene id from the gene names
uniprot_df['b_number'] = uniprot_df['Gene Names'].str.extract(locustag_regex)
uniprot_df = uniprot_df.drop('Gene Names', axis=1)
uniprot_df

/home/samiralvdb/.local/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Entry,Protein names,Length,Mass,Rhea ID,b_number
0,Q88CC1,2-oxoadipate dioxygenase/decarboxylase (EC 1.1...,464,51372,RHEA:71787,PP_5260
1,Q88E10,Methyl-accepting chemotaxis protein McpS,639,68764,NaN,PP_4658
2,Q88E47,"Homogentisate 1,2-dioxygenase (HGDO) (EC 1.13....",433,48011,RHEA:15449,PP_4621
3,Q88F88,Pyoverdine export ATP-binding/permease protein...,654,69687,NaN,PP_4210
4,Q88FF8,Quinone reductase (EC 1.6.5.2) (Chromate reduc...,186,20354,RHEA:46160 RHEA:46164 RHEA:44372 RHEA:44368,PP_4138
...,...,...,...,...,...,...
5524,Q88RW2,Transcriptional regulator,119,13492,NaN,PP_0017
5525,Q88RW3,TniQ family protein,638,71170,NaN,PP_0016
5526,Q88RW4,Transposon protein,319,36304,NaN,PP_0015
5527,Q88RW5,Transposase,652,74345,NaN,PP_0014


### 4.3 Match Uniprot to BIGG data

In [15]:
id_mapper_with_protein = pd.merge(id_mapper_df, uniprot_df, on='b_number', how='left')

id_mapper_with_protein

,rxn_id,kegg_id,rhea_id,mnx_id,Reactants,Products,EC,GPR,b_number,Entry,Protein names,Length,Mass,Rhea ID
0,3HAD160,R04544,NaN,MNXR94881,[C04633],"[[C00001, C01328], C05763]","[2.3.1.85, 2.3.1.86, 4.2.1.61, 4.2.1.59, 2.3.1.-]",PP_1602 or PP_4174,PP_1602,Q88MG9,3-hydroxyacyl-[acyl-carrier-protein] dehydrata...,146.0,16583.0,RHEA:13097
1,3HAD160,R04544,NaN,MNXR94881,[C04633],"[[C00001, C01328], C05763]","[2.3.1.85, 2.3.1.86, 4.2.1.61, 4.2.1.59, 2.3.1.-]",PP_1602 or PP_4174,PP_4174,Q88FC4,3-hydroxydecanoyl-[acyl-carrier-protein] dehyd...,171.0,18791.0,RHEA:13097 RHEA:41860 RHEA:23568
2,1P2CBXLCYCL,NaN,NaN,NaN,[C01110],[],NaN,PP_s0001,NaN,G1JLS8,Phosphatidylcholine synthase (EC 2.7.8.24),205.0,21908.0,NaN
3,1P2CBXLCYCL,NaN,NaN,NaN,[C01110],[],NaN,PP_s0001,NaN,V6C0L5,Proteolysis tag peptide encoded by tmRNA Psmon...,14.0,1529.0,NaN
4,1P2CBXLR,NaN,NaN,NaN,[],"[C00006, [C00148, C16435]]",NaN,PP_3591,PP_3591,Q88GX6,Delta 1-piperideine-2-carboxylate reductase (E...,332.0,35143.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3865,4MBZDH,NaN,NaN,MNXR95031,"[C06758, [C00001, C01328], C00003]","[C00080, C00004, C01454]","[1.2.1.3, 1.2.1.28, 1.2.1.7]",pWW0_131,NaN,V6C0L5,Proteolysis tag peptide encoded by tmRNA Psmon...,14.0,1529.0,NaN
3866,4MCAT23DOX,R05295,NaN,MNXR95032,"[C06730, C00007]","[C00080, C06760]",1.13.11.2,pWW0_097,NaN,G1JLS8,Phosphatidylcholine synthase (EC 2.7.8.24),205.0,21908.0,NaN
3867,4MCAT23DOX,R05295,NaN,MNXR95032,"[C06730, C00007]","[C00080, C06760]",1.13.11.2,pWW0_097,NaN,V6C0L5,Proteolysis tag peptide encoded by tmRNA Psmon...,14.0,1529.0,NaN
3868,4OD,NaN,NaN,MNXR126192,"[C00080, C03453]",[],NaN,pWW0_091,NaN,G1JLS8,Phosphatidylcholine synthase (EC 2.7.8.24),205.0,21908.0,NaN


In [16]:
#get the RHEA ids from the rhea id column and create an individual row for each id
# Split the strings into individual 'RHEA' entries and explode them into separate rows
id_mapper_with_protein['rhea_id_up'] = id_mapper_with_protein['Rhea ID'].str.split(' ')
id_mapper_with_protein = id_mapper_with_protein.explode('rhea_id_up', ignore_index=True)

# Extract the 'RHEA' IDs using a regular expression
id_mapper_with_protein['rhea_id_up'] = id_mapper_with_protein['rhea_id_up'].str.extract(r'RHEA:(\d+)')

# Reset the index to get a clean DataFrame
id_mapper_with_protein_parsed = id_mapper_with_protein.reset_index(drop=True).drop(['Rhea ID','rhea_id', 'mnx_id'], axis =1)

id_mapper_with_protein_parsed

,rxn_id,kegg_id,Reactants,Products,EC,GPR,b_number,Entry,Protein names,Length,Mass,rhea_id_up
0,3HAD160,R04544,[C04633],"[[C00001, C01328], C05763]","[2.3.1.85, 2.3.1.86, 4.2.1.61, 4.2.1.59, 2.3.1.-]",PP_1602 or PP_4174,PP_1602,Q88MG9,3-hydroxyacyl-[acyl-carrier-protein] dehydrata...,146.0,16583.0,13097
1,3HAD160,R04544,[C04633],"[[C00001, C01328], C05763]","[2.3.1.85, 2.3.1.86, 4.2.1.61, 4.2.1.59, 2.3.1.-]",PP_1602 or PP_4174,PP_4174,Q88FC4,3-hydroxydecanoyl-[acyl-carrier-protein] dehyd...,171.0,18791.0,13097
2,3HAD160,R04544,[C04633],"[[C00001, C01328], C05763]","[2.3.1.85, 2.3.1.86, 4.2.1.61, 4.2.1.59, 2.3.1.-]",PP_1602 or PP_4174,PP_4174,Q88FC4,3-hydroxydecanoyl-[acyl-carrier-protein] dehyd...,171.0,18791.0,41860
3,3HAD160,R04544,[C04633],"[[C00001, C01328], C05763]","[2.3.1.85, 2.3.1.86, 4.2.1.61, 4.2.1.59, 2.3.1.-]",PP_1602 or PP_4174,PP_4174,Q88FC4,3-hydroxydecanoyl-[acyl-carrier-protein] dehyd...,171.0,18791.0,23568
4,1P2CBXLCYCL,NaN,[C01110],[],NaN,PP_s0001,NaN,G1JLS8,Phosphatidylcholine synthase (EC 2.7.8.24),205.0,21908.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
4624,4MBZDH,NaN,"[C06758, [C00001, C01328], C00003]","[C00080, C00004, C01454]","[1.2.1.3, 1.2.1.28, 1.2.1.7]",pWW0_131,NaN,V6C0L5,Proteolysis tag peptide encoded by tmRNA Psmon...,14.0,1529.0,NaN
4625,4MCAT23DOX,R05295,"[C06730, C00007]","[C00080, C06760]",1.13.11.2,pWW0_097,NaN,G1JLS8,Phosphatidylcholine synthase (EC 2.7.8.24),205.0,21908.0,NaN
4626,4MCAT23DOX,R05295,"[C06730, C00007]","[C00080, C06760]",1.13.11.2,pWW0_097,NaN,V6C0L5,Proteolysis tag peptide encoded by tmRNA Psmon...,14.0,1529.0,NaN
4627,4OD,NaN,"[C00080, C03453]",[],NaN,pWW0_091,NaN,G1JLS8,Phosphatidylcholine synthase (EC 2.7.8.24),205.0,21908.0,NaN


### 4.3 Use the Rhea ID from uniprot to find the right KEGG ids

In [17]:
rhea2kegg_df = pd.read_csv(RHEA2KEGG_FILE_PATH, sep ='\t')
rhea2kegg_df = rhea2kegg_df.drop(['DIRECTION', 'MASTER_ID'], axis=1)
rhea2kegg_dict = rhea2kegg_df.to_dict()

In [18]:
nmbr_kegg_mapped = 0
for i, row in id_mapper_with_protein_parsed.iterrows():
    if isinstance(row.kegg_id, float):
        if row.rhea_id_up in rhea2kegg_dict.keys():
            id_mapper_with_protein_parsed.kegg_id.iloc[i] = rhea2kegg_dict[row.rhea_id_up]
            nmbr_kegg_mapped += 1
print('additional number of KEGG ids which were mapped were: ', nmbr_kegg_mapped)

additional number of KEGG ids which were mapped were:  0


### 5. Match the BiGG model reactiona and enzymes to the kcats from GotEnzymes
Find first all enzymes for a reaction, then for each enzyme determine if the kcats from GotEnzymes are for a forward or reverse reaction

In [19]:
id_mapper_final = id_mapper_with_protein_parsed
id_mapper_final = id_mapper_final.drop(['rhea_id_up'], axis = 1).rename({'Entry':'uniprot_id'}, axis=1)
id_mapper_final = id_mapper_final.explode('kegg_id', ignore_index=True)
id_mapper_final = id_mapper_final.explode('EC', ignore_index=True)

# id_mapper_final = id_mapper_final.drop_duplicates(subset = ['kegg_id'])
id_mapper_final.head()

,rxn_id,kegg_id,Reactants,Products,EC,GPR,b_number,uniprot_id,Protein names,Length,Mass
0,3HAD160,R04544,[C04633],"[[C00001, C01328], C05763]",2.3.1.85,PP_1602 or PP_4174,PP_1602,Q88MG9,3-hydroxyacyl-[acyl-carrier-protein] dehydrata...,146.0,16583.0
1,3HAD160,R04544,[C04633],"[[C00001, C01328], C05763]",2.3.1.86,PP_1602 or PP_4174,PP_1602,Q88MG9,3-hydroxyacyl-[acyl-carrier-protein] dehydrata...,146.0,16583.0
2,3HAD160,R04544,[C04633],"[[C00001, C01328], C05763]",4.2.1.61,PP_1602 or PP_4174,PP_1602,Q88MG9,3-hydroxyacyl-[acyl-carrier-protein] dehydrata...,146.0,16583.0
3,3HAD160,R04544,[C04633],"[[C00001, C01328], C05763]",4.2.1.59,PP_1602 or PP_4174,PP_1602,Q88MG9,3-hydroxyacyl-[acyl-carrier-protein] dehydrata...,146.0,16583.0
4,3HAD160,R04544,[C04633],"[[C00001, C01328], C05763]",2.3.1.-,PP_1602 or PP_4174,PP_1602,Q88MG9,3-hydroxyacyl-[acyl-carrier-protein] dehydrata...,146.0,16583.0


In [20]:
#get the json file from GotEnzymes and parse into workable format
#read in the json file obtained from gotEnzymes (for 'eco')
eco_enzymes_json = pd.read_json(GOTENZYMES_FILE_PATH)
#convert to a readible format
eco_enzymes = eco_enzymes_json.enzymes.to_list()
eco_enzymes_df = pd.DataFrame(eco_enzymes)

#ensure there is only one ec number per row
eco_enzymes_df['ec_number'] = eco_enzymes_df['ec_number'].str.split(pat=';')
eco_enzymes_df = eco_enzymes_df.explode('ec_number', ignore_index=True)
eco_enzymes_df = eco_enzymes_df.dropna(axis=0, subset=['ec_number'])
eco_enzymes_df = eco_enzymes_df.drop(['organism', 'domain'], axis = 1)
eco_enzymes_df.head()

,gene,reaction_id,ec_number,compound,kcat_values
0,PP_0004,R08701,,C00101,3.6784
1,PP_0004,R08701,,C00037,1.4164
2,PP_0004,R08701,,C03479,3.2723
3,PP_0005,R08701,,C00101,2.4913
4,PP_0005,R08701,,C03479,2.1913


In [21]:
# match BiGG and GotEnzymes based on gene id
eco_enzymes_merged = id_mapper_final.merge(eco_enzymes_df, 
                              left_on =['b_number', 'kegg_id'], 
                              right_on = ['gene', 'reaction_id'],
                             how = 'left').drop(['gene', 'reaction_id'], axis=1).rename({'b_number':'gene'}, axis=1)
eco_enzymes_merged

,rxn_id,kegg_id,Reactants,Products,EC,GPR,gene,uniprot_id,Protein names,Length,Mass,ec_number,compound,kcat_values
0,3HAD160,R04544,[C04633],"[[C00001, C01328], C05763]",2.3.1.85,PP_1602 or PP_4174,PP_1602,Q88MG9,3-hydroxyacyl-[acyl-carrier-protein] dehydrata...,146.0,16583.0,NaN,NaN,NaN
1,3HAD160,R04544,[C04633],"[[C00001, C01328], C05763]",2.3.1.86,PP_1602 or PP_4174,PP_1602,Q88MG9,3-hydroxyacyl-[acyl-carrier-protein] dehydrata...,146.0,16583.0,NaN,NaN,NaN
2,3HAD160,R04544,[C04633],"[[C00001, C01328], C05763]",4.2.1.61,PP_1602 or PP_4174,PP_1602,Q88MG9,3-hydroxyacyl-[acyl-carrier-protein] dehydrata...,146.0,16583.0,NaN,NaN,NaN
3,3HAD160,R04544,[C04633],"[[C00001, C01328], C05763]",4.2.1.59,PP_1602 or PP_4174,PP_1602,Q88MG9,3-hydroxyacyl-[acyl-carrier-protein] dehydrata...,146.0,16583.0,NaN,NaN,NaN
4,3HAD160,R04544,[C04633],"[[C00001, C01328], C05763]",2.3.1.-,PP_1602 or PP_4174,PP_1602,Q88MG9,3-hydroxyacyl-[acyl-carrier-protein] dehydrata...,146.0,16583.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10686,4MBZDH,NaN,"[C06758, [C00001, C01328], C00003]","[C00080, C00004, C01454]",1.2.1.7,pWW0_131,NaN,V6C0L5,Proteolysis tag peptide encoded by tmRNA Psmon...,14.0,1529.0,NaN,NaN,NaN
10687,4MCAT23DOX,R05295,"[C06730, C00007]","[C00080, C06760]",1.13.11.2,pWW0_097,NaN,G1JLS8,Phosphatidylcholine synthase (EC 2.7.8.24),205.0,21908.0,NaN,NaN,NaN
10688,4MCAT23DOX,R05295,"[C06730, C00007]","[C00080, C06760]",1.13.11.2,pWW0_097,NaN,V6C0L5,Proteolysis tag peptide encoded by tmRNA Psmon...,14.0,1529.0,NaN,NaN,NaN
10689,4OD,NaN,"[C00080, C03453]",[],NaN,pWW0_091,NaN,G1JLS8,Phosphatidylcholine synthase (EC 2.7.8.24),205.0,21908.0,NaN,NaN,NaN


### 6. Extract directionalities and gapfill
Reactions which are associated with an enzyme, but not with a kcat from GotEnzymes will be assignes
a default kcat and if required a default molmass.

In [22]:
# Get directionalities and fill in the gaps
eco_enzymes_merged['direction'] = 'f'
for index, row in eco_enzymes_merged.iterrows():
    #there is a kegg id and kcat associated to this reaction
    if not isinstance(row.kcat_values, float):
        if row.compound in row.Products:
            eco_enzymes_merged.direction.iloc[index] = 'b'
        elif not row.compound in row.Reactants:
            print(f'Compound {row["compound"]} is not associated to reaction {row["kegg_id"]}')
            continue
    #there is no kcat associated to this reaction, try to use EC number to get a kcat value
    else: 
        got_enzyme_info = eco_enzymes_df[eco_enzymes_df['ec_number']==row.EC]
        for i, info in got_enzyme_info.iterrows():
            #is there any enzyme association if we look at the metabolites in the reaction and the EC number?
            if info.compound in row.Products:
                direction = 'b'
            elif info.compound in row.Reactants:
                direction = 'f'
            else:
                continue
            #change the kcat if it is not there already or create a new enzyme association
            if np.isnan(row.kcat_values):
                eco_enzymes_merged.direction.iloc[index] = 'b'
                eco_enzymes_merged.kcat_values.iloc[index] = info.kcat_values
            else:
                eco_enzymes_merged.loc[len(eco_enzymes_merged)] = row.to_list()[:-2]+[info.kcat_values, 'b']
        #add default information for unmappable proteins
        if np.isnan(row.kcat_values) & (isinstance(row.uniprot_id, str)) & (row.GPR!= 's0001'):
            eco_enzymes_merged.direction.iloc[index] = 'f'
            eco_enzymes_merged.kcat_values.iloc[index] = DEFAULT_KCAT
        if np.isnan(row.Mass) & (len(row.GPR)>2) & (row.GPR!= 's0001'):
            eco_enzymes_merged.Mass.iloc[index] = DEFAULT_PROT_MW
        
        
eco_enzymes_mapped = eco_enzymes_merged.drop(['ec_number', 'compound'], axis=1).rename({'Mass':'molMass'}, axis =1)
eco_enzymes_mapped

/tmp/ipykernel_48445/3802666525.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'f'
/tmp/ipykernel_48445/3802666525.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.kcat_values.iloc[index] = DEFAULT_KCAT
/tmp/ipykernel_48445/3802666525.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'f'
/tmp/ipykernel_48445/3802666525.

/tmp/ipykernel_48445/3802666525.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'b'
/tmp/ipykernel_48445/3802666525.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.kcat_values.iloc[index] = info.kcat_values
/tmp/ipykernel_48445/3802666525.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'f'
/tmp/ipykernel_48445/3802666

/tmp/ipykernel_48445/3802666525.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'f'
/tmp/ipykernel_48445/3802666525.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.kcat_values.iloc[index] = DEFAULT_KCAT
/tmp/ipykernel_48445/3802666525.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'b'
/tmp/ipykernel_48445/3802666525.

/tmp/ipykernel_48445/3802666525.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'f'
/tmp/ipykernel_48445/3802666525.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.kcat_values.iloc[index] = DEFAULT_KCAT
/tmp/ipykernel_48445/3802666525.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'f'
/tmp/ipykernel_48445/3802666525.

/tmp/ipykernel_48445/3802666525.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'f'
/tmp/ipykernel_48445/3802666525.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.kcat_values.iloc[index] = DEFAULT_KCAT
/tmp/ipykernel_48445/3802666525.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'f'
/tmp/ipykernel_48445/3802666525.

...

/tmp/ipykernel_48445/3802666525.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'f'
/tmp/ipykernel_48445/3802666525.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.kcat_values.iloc[index] = DEFAULT_KCAT
/tmp/ipykernel_48445/3802666525.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'f'
/tmp/ipykernel_48445/3802666525.

/tmp/ipykernel_48445/3802666525.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'f'
/tmp/ipykernel_48445/3802666525.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.kcat_values.iloc[index] = DEFAULT_KCAT
/tmp/ipykernel_48445/3802666525.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'b'
/tmp/ipykernel_48445/3802666525.

/tmp/ipykernel_48445/3802666525.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'f'
/tmp/ipykernel_48445/3802666525.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.kcat_values.iloc[index] = DEFAULT_KCAT
/tmp/ipykernel_48445/3802666525.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'f'
/tmp/ipykernel_48445/3802666525.

/tmp/ipykernel_48445/3802666525.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'f'
/tmp/ipykernel_48445/3802666525.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.kcat_values.iloc[index] = DEFAULT_KCAT
/tmp/ipykernel_48445/3802666525.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'b'
/tmp/ipykernel_48445/3802666525.

,rxn_id,kegg_id,Reactants,Products,EC,GPR,gene,uniprot_id,Protein names,Length,molMass,kcat_values,direction
0,3HAD160,R04544,[C04633],"[[C00001, C01328], C05763]",2.3.1.85,PP_1602 or PP_4174,PP_1602,Q88MG9,3-hydroxyacyl-[acyl-carrier-protein] dehydrata...,146.0,16583.0,11.0000,f
1,3HAD160,R04544,[C04633],"[[C00001, C01328], C05763]",2.3.1.86,PP_1602 or PP_4174,PP_1602,Q88MG9,3-hydroxyacyl-[acyl-carrier-protein] dehydrata...,146.0,16583.0,11.0000,f
2,3HAD160,R04544,[C04633],"[[C00001, C01328], C05763]",4.2.1.61,PP_1602 or PP_4174,PP_1602,Q88MG9,3-hydroxyacyl-[acyl-carrier-protein] dehydrata...,146.0,16583.0,11.0000,f
3,3HAD160,R04544,[C04633],"[[C00001, C01328], C05763]",4.2.1.59,PP_1602 or PP_4174,PP_1602,Q88MG9,3-hydroxyacyl-[acyl-carrier-protein] dehydrata...,146.0,16583.0,11.0000,f
4,3HAD160,R04544,[C04633],"[[C00001, C01328], C05763]",2.3.1.-,PP_1602 or PP_4174,PP_1602,Q88MG9,3-hydroxyacyl-[acyl-carrier-protein] dehydrata...,146.0,16583.0,11.0000,f
...,...,...,...,...,...,...,...,...,...,...,...,...,...
43704,4HTHRS,R05086,"[[C00001, C01328], C06055]","[C06056, C00009]",4.2.3.1,PP_1471 or PP_0662,PP_0662,Q88Q36,Threonine synthase,462.0,50007.0,9.3337,b
43705,4HTHRS,R05086,"[[C00001, C01328], C06055]","[C06056, C00009]",4.2.3.1,PP_1471 or PP_0662,PP_0662,Q88Q36,Threonine synthase,462.0,50007.0,2.2065,b
43706,4HTHRS,R05086,"[[C00001, C01328], C06055]","[C06056, C00009]",4.2.3.1,PP_1471 or PP_0662,PP_0662,Q88Q36,Threonine synthase,462.0,50007.0,2.1743,b
43707,4HTHRS,R05086,"[[C00001, C01328], C06055]","[C06056, C00009]",4.2.3.1,PP_1471 or PP_0662,PP_0662,Q88Q36,Threonine synthase,462.0,50007.0,7.7639,b


In [23]:
#save the dataframe
eco_enzymes_mapped = eco_enzymes_mapped.drop_duplicates(subset = ['rxn_id', 'gene', 'direction'],
                                                       ignore_index=True)
eco_enzymes_mapped.to_excel(AE_OUTPUT_FILE_PATH)


### 7. Save the dataframe to the proper format for building PAMs

In [24]:
# Get the information about the enzyme sectors
unused_enzymes_df = pd.read_excel(PAM_DATA_FILE_PATH,
                                 sheet_name = 'ExcessEnzymes')
translational_protein_df = pd.read_excel(PAM_DATA_FILE_PATH,
                                 sheet_name = 'Translational')

In [25]:
# Save it to a new excel file
with pd.ExcelWriter(OUTPUT_FILE_PATH) as writer:
    eco_enzymes_mapped.to_excel(writer, sheet_name='ActiveEnzymes')
    translational_protein_df.to_excel(writer, sheet_name='Translational')
    unused_enzymes_df.to_excel(writer, sheet_name = 'UnusedEnzyme')